# NB21 · OOP Advanced

`advanced/` · 20 questions · the last one is a story project

Making your own classes behave like built-in types: readable `repr`s,
equality and hashing, operators, containers, properties, alternative
constructors, dataclasses, enums, abstract classes, multiple inheritance
and subclassing built-ins. Every question asks you to **write classes and
use them**. Values that would come from `input()` are given to you as
variables.

## You'll need

- NB14: OOP basics
- NB20: intermediate capstone

## You'll practise

- __repr__, __str__, __format__
- The __eq__ and __hash__ contract; ordering with total_ordering
- Operator overloading: __add__, __radd__, __iadd__, returning NotImplemented
- Container protocol: __len__, __getitem__, __setitem__, __contains__, __iter__; __call__, __bool__
- Properties: getter, setter, validation, computed values
- classmethod alternative constructors; staticmethod
- dataclasses: field(default_factory), frozen, order, __post_init__, slots, kw_only
- enum: Enum, IntEnum, StrEnum, Flag, auto
- Abstract base classes; multiple inheritance, MRO, cooperative super(); mixins; __slots__
- Subclassing built-ins (and why UserDict exists); copy.replace() (3.13)

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Build

Write a class `Point` with `x` and `y` and a `__repr__` that shows text you
could type to recreate the point, like `Point(3, 4)`. Print one point, then
print a list holding two points.

**Expected output:**

```text
Point(3, 4)
[Point(0, 0), Point(-1, 5)]
```

<details><summary><b>Solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

print(Point(3, 4))
print([Point(0, 0), Point(-1, 5)])
```

</details>

In [ ]:
# your code here


### Q02 · Output-match

Write a class `Price` that stores an `amount`. Make it work in f-strings:
the format spec `eur` shows two decimals followed by a space and `EUR`;
any other spec is applied to the number itself. Print the same price once
with `eur` and once with the spec `010.3f`.

**Expected output:**

```text
12.50 EUR
000012.500
```

<details><summary><b>Solution</b></summary>

```python
class Price:
    def __init__(self, amount):
        self.amount = amount

    def __format__(self, spec):
        if spec == "eur":
            return f"{self.amount:.2f} EUR"
        return format(self.amount, spec)

p = Price(12.5)
print(f"{p:eur}")
print(f"{p:010.3f}")
```

</details>

In [ ]:
# your code here


### Q03 · Puzzle

Tags compare **ignoring case**: write a class `Tag` where `Tag("Python")`
equals `Tag("python")`. Build a `Tag` for each name, put them all in a
`set`, and print the size of the set and whether `Tag("PYTHON")` is in it.

**Expected output:**

```text
3
True
```

<details><summary><b>Solution</b></summary>

```python
class Tag:
    def __init__(self, name):
        self.name = name.lower()

    def __eq__(self, other):
        return isinstance(other, Tag) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

tags = {Tag(n) for n in names}
print(len(tags))
print(Tag("PYTHON") in tags)
```

</details>

In [ ]:
names = ["Python", "python", "Code", "CODE", "Bugs"]

# your code here


### Q04 · Real-life

Write a class `Thermometer` whose `celsius` is a **property**: setting it
below -273.15 raises `ValueError("below absolute zero")`. Add a read-only
`fahrenheit` property computed from it. Create one at 25, print its
Fahrenheit value, then try to set -300, print the error message and print
that the temperature is still 25.

**Expected output:**

```text
77.0
Rejected: below absolute zero
25
```

<details><summary><b>Solution</b></summary>

```python
class Thermometer:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Thermometer(25)
print(t.fahrenheit)
try:
    t.celsius = -300
except ValueError as error:
    print("Rejected:", error)
print(t.celsius)
```

</details>

In [ ]:
# your code here


### Q05 · Real-life

A ticket queue at a help desk: write `TicketQueue` holding a list of
waiting names (passed to `__init__`) so that `len(queue)` is the number
waiting. Print the length of a queue with three people, then print whether
an empty queue counts as `True` or `False` in a condition (use `bool()`).

**Expected output:**

```text
3
False
```

<details><summary><b>Solution</b></summary>

```python
class TicketQueue:
    def __init__(self, waiting):
        self.waiting = list(waiting)

    def __len__(self):
        return len(self.waiting)

print(len(TicketQueue(["Ana", "Ben", "Cy"])))
print(bool(TicketQueue([])))
```

</details>

In [ ]:
# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

A game moves things around with vectors. Write `Vector` with `x` and `y`,
a `__repr__` like `Vector(4, 6)`, `+` between two vectors and `*` by a
number. Print `a + b` and `a * 3`.

**Expected output:**

```text
Vector(4, 6)
Vector(3, 6)
```

<details><summary><b>Solution</b></summary>

```python
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, factor):
        return Vector(self.x * factor, self.y * factor)

a = Vector(1, 2)
b = Vector(3, 4)
print(a + b)
print(a * 3)
```

</details>

In [ ]:
# your code here


### Q07 · Puzzle

Write `Cash` (an `amount` in euros, `__repr__` like `Cash(12)`) whose `+`
works only between two `Cash` objects. For anything else `__add__` must
**return** the special value that lets Python raise its normal `TypeError`.
Print `Cash(5) + Cash(7)`, then try `Cash(5) + "x"` and print
`cannot add` when that fails.

**Expected output:**

```text
Cash(12)
cannot add
```

<details><summary><b>Solution</b></summary>

```python
class Cash:
    def __init__(self, amount):
        self.amount = amount

    def __repr__(self):
        return f"Cash({self.amount})"

    def __add__(self, other):
        if not isinstance(other, Cash):
            return NotImplemented
        return Cash(self.amount + other.amount)

print(Cash(5) + Cash(7))
try:
    Cash(5) + "x"
except TypeError:
    print("cannot add")
```

</details>

In [ ]:
# your code here


### Q08 · Real-life

A teacher wants to call `sum()` on a list of `Score` objects (each has a
`value`, and `__repr__` like `Score(9)`). Write `Score` so that `+` gives a
new `Score` and `sum(scores)` works, even though `sum` starts from the
plain number 0. Print the total of the given scores.

**Expected output:**

```text
Score(20)
```

<details><summary><b>Solution</b></summary>

```python
class Score:
    def __init__(self, value):
        self.value = value

    def __repr__(self):
        return f"Score({self.value})"

    def __add__(self, other):
        if not isinstance(other, Score):
            return NotImplemented
        return Score(self.value + other.value)

    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

scores = [Score(4), Score(9), Score(7)]
print(sum(scores))
```

</details>

In [ ]:
# your code here


### Q09 · Real-life

Write `Inventory` wrapping a dict of stock counts, so a shop can use it
like a container: `inv["apple"] = 5` stores, `inv["apple"]` reads,
`"apple" in inv` tests, `len(inv)` counts the products and a `for` loop
over it visits the product names. Its truth value must be `True` only if
some product has stock above 0. Set three products, loop over it, print
the length and a membership test, then print the truth value of an
inventory whose products are all at 0.

**Expected output:**

```text
apple 5
pear 0
fig 12
3 True False
False
```

<details><summary><b>Solution</b></summary>

```python
class Inventory:
    def __init__(self):
        self._stock = {}

    def __setitem__(self, name, count):
        self._stock[name] = count

    def __getitem__(self, name):
        return self._stock[name]

    def __contains__(self, name):
        return name in self._stock

    def __len__(self):
        return len(self._stock)

    def __iter__(self):
        return self._stock.__iter__()

    def __bool__(self):
        return any(count > 0 for count in self._stock.values())

inv = Inventory()
inv["apple"] = 5
inv["pear"] = 0
inv["fig"] = 12
for name in inv:
    print(name, inv[name])
print(len(inv), "pear" in inv, "kiwi" in inv)
empty = Inventory()
empty["apple"] = 0
print(bool(empty))
```

</details>

In [ ]:
# your code here


### Q10 · Real-life

Write a class `VAT` that is built with a rate and can then be **called like
a function**: `vat(10.0)` returns the price with VAT added, rounded to 2
decimals. Use a 20% rate and apply it to every price in the list with
`map`; print the results as a list.

**Expected output:**

```text
[12.0, 30.6, 119.99]
```

<details><summary><b>Solution</b></summary>

```python
class VAT:
    def __init__(self, rate):
        self.rate = rate

    def __call__(self, price):
        return round(price * (1 + self.rate), 2)

vat = VAT(0.20)
print(list(map(vat, prices)))
```

</details>

In [ ]:
prices = [10.0, 25.5, 99.99]

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

Write a class `Duration` that stores total `minutes`. Give it a
**classmethod** `from_text` that builds one from text like `"2h 15m"`, a
**staticmethod** `pad(n)` returning a number as two digits (`5` becomes
`"05"`), a computed property `hours` (minutes divided by 60, rounded to 2
decimals) and a `__str__` that shows `HH:MM`, using `pad`. Build durations
from the given texts and print each one with its hours.

**Expected output:**

```text
02:15 2.25
00:45 0.75
10:05 10.08
```

<details><summary><b>Solution</b></summary>

```python
class Duration:
    def __init__(self, minutes):
        self.minutes = minutes

    @classmethod
    def from_text(cls, text):
        hours, minutes = text.split()
        return cls(int(hours[:-1]) * 60 + int(minutes[:-1]))

    @staticmethod
    def pad(n):
        return str(n).zfill(2)

    @property
    def hours(self):
        return round(self.minutes / 60, 2)

    def __str__(self):
        return Duration.pad(self.minutes // 60) + ":" + Duration.pad(self.minutes % 60)

for text in texts:
    d = Duration.from_text(text)
    print(d, d.hours)
```

</details>

In [ ]:
texts = ["2h 15m", "0h 45m", "10h 5m"]

# your code here


### Q12 · Real-life

Version numbers must sort by their numbers, not as text (`"1.10"` is newer
than `"1.9"`). Write `Version` built from text like `"1.10.2"`, with `==`
and `<` defined by hand and the other comparisons filled in with the
`total_ordering` decorator. `str(version)` gives the original text. Print
the given versions sorted from oldest to newest, then the newest one.

**Expected output:**

```text
['0.9.9', '1.2.11', '1.9.0', '1.10.0', '1.10.2']
1.10.2
```

<details><summary><b>Solution</b></summary>

```python
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, text):
        self.text = text
        self.parts = tuple(int(p) for p in text.split("."))

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts

    def __str__(self):
        return self.text

versions = [Version(t) for t in texts]
print([str(v) for v in sorted(versions)])
print(max(versions))
```

</details>

In [ ]:
texts = ["1.9.0", "1.10.2", "1.2.11", "0.9.9", "1.10.0"]

# your code here


### Q13 · Real-life

Write a dataclass `Student` using `slots=True` and `kw_only=True`, with a
`name`, a list of `grades` that starts empty for every student (no shared
list!) and a `__post_init__` that turns the name into title case. Create
two students from the given names, add a grade to the first only, print
both, and print whether their `grades` lists are the same object.

**Expected output:**

```text
Student(name='Ada Lovelace', grades=[90])
Student(name='Alan Turing', grades=[])
False
```

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass, field


@dataclass(slots=True, kw_only=True)
class Student:
    name: str
    grades: list = field(default_factory=list)

    def __post_init__(self):
        self.name = self.name.title()

a = Student(name=name_a)
b = Student(name=name_b)
a.grades.append(90)
print(a)
print(b)
print(a.grades is b.grades)
```

</details>

In [ ]:
name_a = "ada lovelace"
name_b = "alan turing"

# your code here


### Q14 · Real-life

A calendar needs appointments that can be sorted, de-duplicated and never
changed by accident. Write a **frozen**, **ordered** dataclass `Slot` with
`day`, `hour` and `title` (in that order). From the given list, print the
sorted list of unique slots. Then use `copy.replace` to make a copy of the
first slot of the original list moved to hour 16 and print it, and print
`frozen` when assigning to a field of a slot fails.

**Expected output:**

```text
[Slot(day=1, hour=9, title='standup'), Slot(day=2, hour=10, title='gym'), Slot(day=2, hour=14, title='dentist')]
Slot(day=2, hour=16, title='dentist')
frozen
```

<details><summary><b>Solution</b></summary>

```python
import copy
from dataclasses import dataclass, FrozenInstanceError


@dataclass(frozen=True, order=True)
class Slot:
    day: int
    hour: int
    title: str

slots = [Slot(*b) for b in bookings]
print(sorted(set(slots)))
print(copy.replace(slots[0], hour=16))
try:
    slots[0].hour = 1
except FrozenInstanceError:
    print("frozen")
```

</details>

In [ ]:
from dataclasses import dataclass

bookings = [(2, 14, "dentist"), (1, 9, "standup"), (2, 10, "gym"), (1, 9, "standup")]

# your code here


### Q15 · Real-life

A to-do app. Write an `IntEnum` `Priority` (LOW=1, MEDIUM=2, HIGH=3) and a
`StrEnum` `Status` with members `TODO` and `DONE` whose values are filled
in by `auto()`. Turn the priority names in `tasks` into `Priority`
members, print the tasks from most to least important (ties keep their
original order) as `HIGH: write report`, then print whether
`Status.DONE == "done"`.

**Expected output:**

```text
HIGH: write report
HIGH: pay rent
MEDIUM: call mum
LOW: water plants
True
```

<details><summary><b>Solution</b></summary>

```python
from enum import IntEnum, StrEnum, auto


class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3


class Status(StrEnum):
    TODO = auto()
    DONE = auto()


ordered = sorted(tasks, key=lambda task: Priority[task[1]], reverse=True)
for title, level in ordered:
    print(Priority[level].name + ": " + title)
print(Status.DONE == "done")
```

</details>

In [ ]:
tasks = [("write report", "HIGH"), ("water plants", "LOW"), ("pay rent", "HIGH"), ("call mum", "MEDIUM")]

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Puzzle

The diamond. Write classes `A`, `B`, `C` and `D` where `B` and `C` both
inherit from `A`, and `D` inherits from `B` then `C`. Each class has a
method `setup(log)` that appends its own class name to `log`, then
cooperatively calls `super().setup(log)` (except `A`, which only
appends). Print the class names in the method resolution order of `D`, then
call `D().setup(log)` on an empty list and print the log.

**Expected output:**

```text
['D', 'B', 'C', 'A', 'object']
['D', 'B', 'C', 'A']
```

<details><summary><b>Solution</b></summary>

```python
class A:
    def setup(self, log):
        log.append("A")


class B(A):
    def setup(self, log):
        log.append("B")
        super().setup(log)


class C(A):
    def setup(self, log):
        log.append("C")
        super().setup(log)


class D(B, C):
    def setup(self, log):
        log.append("D")
        super().setup(log)


print([cls.__name__ for cls in D.__mro__])
log = []
D().setup(log)
print(log)
```

</details>

In [ ]:
# your code here


### Q17 · Real-life

A drawing app. Write an abstract base class `Shape` (use `ABC`) with an
abstract method `area()` and a normal method `describe()` returning
`"<ClassName> with area <area rounded to 2 decimals>"`. Add `Circle(radius)`
(use 3.14159 for pi) and `Square(side)`. Print that `Shape()` cannot be
created (catch the `TypeError`, print `abstract`), then print the
description of every shape in the given list.

**Expected output:**

```text
abstract
Circle with area 12.57
Square with area 9
Circle with area 0.79
```

<details><summary><b>Solution</b></summary>

```python
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

    def describe(self):
        return type(self).__name__ + " with area " + str(round(self.area(), 2))


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


try:
    Shape()
except TypeError:
    print("abstract")
for kind, size in sizes:
    shape = Circle(size) if kind == "circle" else Square(size)
    print(shape.describe())
```

</details>

In [ ]:
sizes = [("circle", 2), ("square", 3), ("circle", 0.5)]

# your code here


### Q18 · Puzzle

File permissions as flags. Write a `Flag` enum `Perm` with `READ`, `WRITE`
and `EXEC` (use `auto()`). A user starts with `READ | WRITE`. Print whether
`WRITE` and `EXEC` are in the user's permissions, then give the user
`EXEC` and take `WRITE` away (with `|=` and `&= ~`). Finally print the
result as a string of `r`, `w`, `x` with `-` for a missing flag.

**Expected output:**

```text
True False
r-x
```

<details><summary><b>Solution</b></summary>

```python
from enum import Flag, auto


class Perm(Flag):
    READ = auto()
    WRITE = auto()
    EXEC = auto()


user = Perm.READ | Perm.WRITE
print(Perm.WRITE in user, Perm.EXEC in user)
user |= Perm.EXEC
user &= ~Perm.WRITE
letters = ""
for perm, letter in zip(Perm, "rwx"):
    letters += letter if perm in user else "-"
print(letters)
```

</details>

In [ ]:
# your code here


### Q19 · Real-life

HTTP header names ignore case. Subclass `UserDict` (not `dict`) into
`Headers`, so every way of storing, reading and testing a key lower-cases
it first: `h["Content-Type"] = ...`, `h["content-type"]`, `"ACCEPT" in h`
and `h.update({...})` must all agree. Store a content type, update with an
upper-case `ACCEPT` key, then print the content type read with a different
case, whether `"Accept"` is in the headers, and the sorted keys.

**Expected output:**

```text
text/html
True
['accept', 'content-type']
```

<details><summary><b>Solution</b></summary>

```python
from collections import UserDict


class Headers(UserDict):
    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)

    def __getitem__(self, key):
        return super().__getitem__(key.lower())

    def __contains__(self, key):
        return super().__contains__(key.lower())


h = Headers()
h["Content-Type"] = "text/html"
h.update({"ACCEPT": "*/*"})
print(h["content-TYPE"])
print("Accept" in h)
print(sorted(h))
```

</details>

In [ ]:
# your code here


---

## Q20 · Story project: Money Library

The *Globe Shop* sells in three currencies and keeps messing up its totals
with floats and loose numbers. You will build a tiny money library: a
currency enum, a safe `Money` type with operators, a converter and a
shopping basket. Each step builds on the one before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
RATES = {"EUR": "1", "USD": "0.92", "GBP": "1.17"}   # value of 1 unit in EUR
purchases = [("coffee beans", "12.50", "EUR"), ("headphones", "89.99", "USD"),
             ("book", "15.00", "GBP"), ("stickers", "3.20", "EUR")]


### Step 1

Create a `StrEnum` `Currency` with members `EUR`, `USD` and `GBP` (values are the same text). Print the list of values, then try `Currency("JPY")` and print `unknown currency` when it fails.

**Expected output:**

```text
['EUR', 'USD', 'GBP']
unknown currency
```

<details><summary><b>Solution</b></summary>

```python
from decimal import Decimal
from enum import StrEnum


class Currency(StrEnum):
    EUR = "EUR"
    USD = "USD"
    GBP = "GBP"


print([c.value for c in Currency])
try:
    Currency("JPY")
except ValueError:
    print("unknown currency")
```

</details>

In [ ]:
# your code here


### Step 2

Write a frozen dataclass `Money(amount, currency)` where `amount` must be a `Decimal` (otherwise `TypeError`). Support `+` (same currency only, else `ValueError`; other types return `NotImplemented`), `__radd__` so `sum()` works, `*` by an `int`, `<` (plus `total_ordering`) and a `__str__` like `12.50 EUR`. Print a sum of two, the sum of a list, a product, the larger of two, and the two error messages' types.

**Expected output:**

```text
15.70 EUR
28.20 EUR
9.60 EUR
12.50 EUR
ValueError
TypeError
```

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass
from functools import total_ordering


@total_ordering
@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: Currency

    def __post_init__(self):
        if not isinstance(self.amount, Decimal):
            raise TypeError("amount must be a Decimal")

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("currency mismatch")
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        return self if other == 0 else NotImplemented

    def __mul__(self, factor):
        if not isinstance(factor, int):
            return NotImplemented
        return Money(self.amount * factor, self.currency)

    def __lt__(self, other):
        return self.amount < other.amount

    def __str__(self):
        return f"{self.amount:.2f} {self.currency}"


a = Money(Decimal("12.50"), Currency.EUR)
b = Money(Decimal("3.20"), Currency.EUR)
print(a + b)
print(sum([a, b, a]))
print(b * 3)
print(max(a, b))
for bad in (lambda: a + Money(Decimal("1"), Currency.USD), lambda: Money(1.5, Currency.EUR)):
    try:
        bad()
    except (ValueError, TypeError) as error:
        print(type(error).__name__)
```

</details>

In [ ]:
# your code here


### Step 3

Write `convert(money, target)` that changes a `Money` into the target currency using `RATES` (value in EUR, as `Decimal`), rounded to 2 decimals with `quantize(Decimal("0.01"))`. Build `items`, a list of `(name, Money)` from `purchases`, and print every item converted to EUR as `name: amount`.

**Expected output:**

```text
coffee beans: 12.50 EUR
headphones: 82.79 EUR
book: 17.55 EUR
stickers: 3.20 EUR
```

<details><summary><b>Solution</b></summary>

```python
def convert(money, target):
    in_eur = money.amount * Decimal(RATES[money.currency])
    amount = (in_eur / Decimal(RATES[target])).quantize(Decimal("0.01"))
    return Money(amount, target)


items = [(name, Money(Decimal(price), Currency(code))) for name, price, code in purchases]
for name, money in items:
    print(name + ": " + str(convert(money, Currency.EUR)))
```

</details>

In [ ]:
# your code here


### Step 4

Write a `SummaryMixin` whose `summary()` returns `<ClassName>: <n> items, total <total>`, and a `Basket(SummaryMixin)` that behaves like a container of `name -> Money`: `[]` read and write, `in`, `len`, and a `total` property (everything converted to EUR and summed, starting from zero euros). Fill a basket from `items` and print its summary, `"book" in basket` and `basket["book"]`.

**Expected output:**

```text
Basket: 4 items, total 116.04 EUR
True
15.00 GBP
```

<details><summary><b>Solution</b></summary>

```python
class SummaryMixin:
    def summary(self):
        return f"{type(self).__name__}: {len(self)} items, total {self.total}"


class Basket(SummaryMixin):
    def __init__(self):
        self.items = {}

    def __setitem__(self, name, money):
        self.items[name] = money

    def __getitem__(self, name):
        return self.items[name]

    def __contains__(self, name):
        return name in self.items

    def __len__(self):
        return len(self.items)

    @property
    def total(self):
        zero = Money(Decimal("0"), Currency.EUR)
        return sum((convert(m, Currency.EUR) for m in self.items.values()), zero)


basket = Basket()
for name, money in items:
    basket[name] = money
print(basket.summary())
print("book" in basket)
print(basket["book"])
```

</details>

In [ ]:
# your code here


### Step 5

Print the shop's receipt: the items sorted from most to least expensive **in EUR**, one per line as `name: <original price> = <price in EUR>`, then the line `TOTAL: <total>`.

**Expected output:**

```text
headphones: 89.99 USD = 82.79 EUR
book: 15.00 GBP = 17.55 EUR
coffee beans: 12.50 EUR = 12.50 EUR
stickers: 3.20 EUR = 3.20 EUR
TOTAL: 116.04 EUR
```

<details><summary><b>Solution</b></summary>

```python
ranked = sorted(basket.items.items(), key=lambda pair: convert(pair[1], Currency.EUR), reverse=True)
for name, money in ranked:
    print(f"{name}: {money} = {convert(money, Currency.EUR)}")
print("TOTAL:", basket.total)
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB22 · Iterators Generators** in `advanced/`.